In [ ]:
import { PGlite } from "npm:@electric-sql/pglite";
import { pg_trgm } from "npm:@electric-sql/pglite/contrib/pg_trgm";
import { unaccent } from "npm:@electric-sql/pglite/contrib/unaccent";

const client = new PGlite(
  process.env.test ? "memory://" : "./../../web/.data/database",
  {
    extensions: { pg_trgm, unaccent },
  },
);


In [ ]:
import { cert, getApp, getApps, initializeApp } from "npm:firebase-admin/app";
import { getFirestore } from "npm:firebase-admin/firestore";

const serviceAccountText = await Deno.readTextFile("./../env/firebase.json");
const serviceAccount = JSON.parse(serviceAccountText);

const app = getApps().length === 0
  ? initializeApp({ credential: cert(serviceAccount) })
  : getApp();

const firestore = getFirestore(app);


In [ ]:
const clubsCollection = firestore.collection("clubs");
const clubsSnapshot = await clubsCollection.get();


In [ ]:
import { stringify } from "jsr:@std/csv";

const clubsData = clubsSnapshot.docs.map((doc) => ({
  uid: doc.id,
  name: doc.data().name,
  isActive: doc.data().active,
  league: doc.data().zdp ? "junior" : "senior",
  region: null,
}));

const csvData = stringify(clubsData, {
  columns: ["uid", "name", "isActive", "league", "region"],
});

await Deno.mkdir("./../data", { recursive: true });
await Deno.writeTextFile("./../data/clubs.csv", csvData);


In [ ]:
const usersCollection = firestore.collection("users");
// Get the data sorted by createdAt in ascending order
const usersSnapshot = await usersCollection.orderBy("createdAt", "asc").get();


In [ ]:
import { stringify } from "jsr:@std/csv";

const usersData = usersSnapshot.docs.map((doc) => {
  const data = doc.data();

  const birthDate = data.birthdate?.toDate();

  return {
    uid: doc.id,
    name: data.name ?? "",
    surname: data.surname ?? "",
    role: data.role ?? "user",
    email: data.email ?? "",
    phone: data.phone ?? "",
    birthDate: birthDate
      ? birthDate.toLocaleDateString("sv-SE", {
        timeZone: "Europe/Bratislava",
      })
      : "",
    createdAt: data.createdAt?.toDate()?.toISOString() ?? "",
    clubId: data.club?.id ?? "",
    seasons: data.seasons
      ? data.seasons
        .map((season: { year: number | string }) => season.year)
        .join(";")
      : "",
    clubManager: data.clubManager ?? false,
    address: data.address ?? "",
    streetAddress: "",
    postalCode: data.postalCode ?? "",
    city: data.city ?? "",
  };
});

const usersCsvData = stringify(usersData, {
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

await Deno.writeTextFile("./../data/users.csv", usersCsvData);


In [ ]:
const hashesCollection = firestore.collection("hashes");
const hashesSnapshot = await hashesCollection.get();


In [ ]:
// Go through each document in the hashes collection and if the group them by the email field. Keep only the latest document for each email (createdAt field).

const hashesData = hashesSnapshot.docs.map((doc) => ({
  uid: doc.id,
  email: doc.data().email ?? "",
  hash: doc.data().hash ?? "",
  createdAt: doc.data().createdAt?.toDate()?.toISOString() ?? "",
}));

const latestHashesMap = new Map<string, any>();

for (const hash of hashesData) {
  const existingHash = latestHashesMap.get(hash.email);

  if (
    !existingHash || new Date(hash.createdAt) > new Date(existingHash.createdAt)
  ) {
    latestHashesMap.set(hash.email, hash);
  }
}

const latestHashesData = Array.from(latestHashesMap.values());


In [ ]:
// Read the file clubs_x.csv
import { parse } from "jsr:@std/csv";

type ClubX = {
  id: number | null;
  uid: string;
  name: string;
  isActive: boolean;
  league: "junior" | "senior" | "university" | null;
  region: "western" | "central" | "eastern" | null;
};

const clubsXFile = await Deno.readTextFile("./../data/clubs_x.csv");
const clubsXData: ClubX[] = await parse(clubsXFile, {
  skipFirstRow: true,
  strip: true,
  columns: ["uid", "name", "isActive", "league", "region"],
});

// Add the clubxdata to the pglite client and map generated ids to the clubs x  using returning data.
clubsXData.forEach(async (club) => {
  await client.query(
    "INSERT INTO clubs (name, is_active, league, region) VALUES ($1, $2, $3, $4) RETURNING id",
    [club.name, club.isActive, club.league, club.region],
  ).then((result) => {
    const generatedId = result.rows[0].id;
    clubsXData.find((c) => c.uid === club.uid)!.id = generatedId;
    console.log(`Inserted club ${club.name} with generated id ${generatedId}`);
  }).catch((error) => {
    console.error(`Error inserting club ${club.name}:`, error);
  });
});


In [ ]:
type UserX = {
  id: number | null;
  uid: string;
  name: string;
  surname: string;
  role: "user" | "admin" | "superadmin" | null;
  email: string;
  phone: string;
  birthDate: string | null;
  createdAt: string | null;
  clubId: number | null;
  seasons: string | null;
  clubManager: boolean | null;
  address: string | null;
  streetAddress: string | null;
  postalCode: string | null;
  city: string | null;
};

// const usersXData: UserX[] = await parse("./../data/users_x.csv");
const usersXFile = await Deno.readTextFile("./../data/users_x.csv");
const usersXData: UserX[] = await parse(usersXFile, {
  skipFirstRow: true,
  strip: true,
  columns: [
    "uid",
    "name",
    "surname",
    "role",
    "email",
    "phone",
    "birthDate",
    "createdAt",
    "clubId",
    "seasons",
    "clubManager",
    "address",
    "streetAddress",
    "postalCode",
    "city",
  ],
});

// Go trough each user in the usersXData and if the user has role "coach", change it to "user" and set the clubManager field to true.
usersXData.forEach((user) => {
  if (user.role === "coach") {
    user.role = "user";
    user.clubManager = true;
  }
  user.postalCode = user.postalCode?.replace(/\s/g, "") || null;
  // If the phone number starts with 09, replace it with +4219
  if (user.phone?.startsWith("09")) {
    user.phone = "+421" + user.phone.substring(1);
  }

  // Remove all non-digit or non-plus characters from the phone number
  user.phone = user.phone?.replace(/[^\d+]/g, "") || null;

  // Get the birthDate from the usersData
  user.birthDate = usersData.find((u) => u.uid === user.uid)?.birthDate || null;
});

// In usersXData, replace all the empty strings with null values.
usersXData.forEach((user) => {
  Object.keys(user).forEach((key) => {
    if (user[key as keyof UserX] === "") {
      user[key as keyof UserX] = null;
    }
  });
});

const removedEmails = new Set<string>();

// Remove all users from usersXData that have name or surname or email as null.
usersXData.forEach((user, index) => {
  if (!user.name || !user.surname || !user.email) {
    console.log(
      `Removing user with uid ${user.uid} because of missing name, surname or email`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Remove all users from usersXData that have birthdate and address as null.
usersXData.forEach((user, index) => {
  if (!user.birthDate && !user.address) {
    console.log(
      `Removing user with uid ${user.uid} because of missing birthdate and address`,
    );
    removedEmails.add(user.email);
    usersXData.splice(index, 1);
  }
});

// Add the usersxdata to the pglite client and map generated ids to the users x using returning data.
usersXData.forEach(async (user) => {
  let generatedId: number | null = null;

  await client.query(
    "INSERT INTO users (name, surname, role, email, phone, birth_date, created_at, street, postal_code, town, email_verified) VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9, $10, true) RETURNING id",
    [
      user.name,
      user.surname,
      user.role,
      user.email,
      user.phone,
      user.birthDate,
      user.createdAt,
      user.streetAddress,
      user.postalCode,
      user.city,
    ],
  ).then((result) => {
    generatedId = result.rows[0].id;
    usersXData.find((u) => u.uid === user.uid)!.id = generatedId;
    console.log(
      `Inserted user ${user.name} ${user.surname} with generated id ${generatedId}`,
    );
  }).catch((error) => {
    generatedId = null;
    removedEmails.add(user.email);
    console.error(`Error inserting user ${user.name} ${user.surname}:`, error);
  });

  // If the user has a clubId, for each season separated by ;, insert a row into the club_memberships table with the generated user id, the club id, and the season year.
  if (user.clubId && user.seasons && generatedId) {
    const seasons = user.seasons.split(";");
    seasons.forEach(async (season) => {
      const mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
      if (!mappedClubId) {
        console.error(
          `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
        );
        return;
      }

      // If user is younger than 14 years old in the given season, set the registration_type to "junior_student", < 19 "senior)student", < 30 "graduate" >= 30 "teacher"
      const birthYear = new Date(user.birthDate ?? "").getFullYear();
      const seasonYear = parseInt(season);
      let registrationType:
        | "junior_student"
        | "senior_student"
        | "graduate"
        | "teacher" = "teacher";

      if (birthYear && seasonYear) {
        const age = seasonYear - birthYear;
        if (age < 14) {
          registrationType = "junior_student";
        } else if (age < 19) {
          registrationType = "senior_student";
        } else if (age < 30) {
          registrationType = "graduate";
        } else {
          registrationType = "teacher";
        }
      }

      await client.query(
        "INSERT INTO club_memberships (user_id, club_id, season, confirmed, registration_type) VALUES ($1, $2, $3, true, $4)",
        [generatedId, mappedClubId, season, registrationType],
      ).then(() => {
        console.log(
          `Inserted club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting club membership for user ${user.name} ${user.surname} in club ${user.clubId} for season ${season}:`,
          error,
        );
      });
    });
  }

  // If clubManager is true, insert a row into the club_managers table with the generated user id and the mapped club id.
  if (
    user.clubManager &&
    (user.clubManager === true || user.clubManager.toLowerCase() === "true") &&
    user.clubId && generatedId
  ) {
    const mappedClubId = clubsXData.find((c) => c.uid === user.clubId)?.id;
    if (!mappedClubId) {
      console.error(
        `Error: Could not find mapped club id for user ${user.name} ${user.surname} with club uid ${user.clubId}`,
      );
      return;
    }
    await client.query(
      "INSERT INTO club_managers (user_id, club_id) VALUES ($1, $2)",
      [generatedId, mappedClubId],
    ).then(() => {
      console.log(
        `Inserted club manager for user ${user.name} ${user.surname} in club ${user.clubId}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting club manager for user ${user.name} ${user.surname} in club ${user.clubId}:`,
        error,
      );
    });
  }

  // If the user has a hash, insert a row into the accounts table with the generated user id and the hash.
  const userHash = latestHashesData.find((h) => h.email === user.email);
  if (userHash && generatedId !== null) {
    await client.query(
      "INSERT INTO accounts (user_id, provider_id, issuer, account_id, password) VALUES ($1, 'credential', 'local:credential', $1, $2)",
      [generatedId, userHash.hash],
    ).then(() => {
      console.log(
        `Inserted account for user ${user.name} ${user.surname}`,
      );
    }).catch((error) => {
      console.error(
        `Error inserting account for user ${user.name} ${user.surname}:`,
        error,
      );
    });
  }

  // In usersSnapshot, check if user has supervisor and supervisorEmail fields. If so, insert a row into the legal_guardians table with the generated user id, the supervisor name, and the supervisor email.
  const userSnapshot = usersSnapshot.docs.find((doc) => doc.id === user.uid);
  if (userSnapshot) {
    const userData = userSnapshot.data();
    if (
      userData.supervisor && userData.supervisorEmail && generatedId !== null
    ) {
      await client.query(
        "INSERT INTO legal_guardians (user_id, name, email) VALUES ($1, $2, $3)",
        [generatedId, userData.supervisor, userData.supervisorEmail],
      ).then(() => {
        console.log(
          `Inserted legal guardian for user ${user.name} ${user.surname}`,
        );
      }).catch((error) => {
        console.error(
          `Error inserting legal guardian for user ${user.name} ${user.surname}:`,
          error,
        );
      });
    }
  }
});

// Write the removed emails to a file removed_emails.txt
await Deno.writeTextFile(
  "./../data/removed_emails.txt",
  Array.from(removedEmails).join("\n"),
);


In [ ]:
const eventsCollection = firestore.collection("events");
const eventsSnapshot = await eventsCollection.get();


In [ ]:
const formatRange = (beginning: Date, end: Date): string => {
  const start = beginning <= end ? beginning : end;
  const finish = beginning <= end ? end : beginning;

  const sameYear = start.getFullYear() === finish.getFullYear();
  const sameMonth = sameYear && start.getMonth() === finish.getMonth();
  const sameDay = sameMonth && start.getDate() === finish.getDate();

  const day = (date: Date) => `${date.getDate()}.`;
  const month = (date: Date) => `${date.getMonth() + 1}.`;
  const year = (date: Date) => `${date.getFullYear()}`;

  if (sameDay) {
    return `${day(start)} ${month(start)} ${year(start)}`;
  }

  if (sameMonth) {
    return `${day(start)} - ${day(finish)} ${month(start)} ${year(start)}`;
  }

  if (sameYear) {
    return `${day(start)} ${month(start)} - ${day(finish)} ${month(finish)} ${
      year(start)
    }`;
  }

  return `${day(start)} ${month(start)} ${year(start)} - ${day(finish)} ${
    month(finish)
  } ${year(finish)}`;
};

const getTargetLeague = (
  eventId: string,
): "junior" | "senior" | "university" | null => {
  if (eventId.startsWith("s") || eventId.startsWith("fsdl")) {
    return "senior";
  }

  if (eventId.startsWith("z") || eventId.startsWith("fjdl")) {
    return "junior";
  }

  return null;
};

const getTargetRegion = (
  eventName: string,
): "western" | "central" | "eastern" | null => {
  const lowerCaseName = eventName.toLowerCase();

  if (lowerCaseName.includes("západ")) {
    return "western";
  }

  if (lowerCaseName.includes("stred")) {
    return "central";
  }

  if (lowerCaseName.includes("východ")) {
    return "eastern";
  }

  return null;
};


In [ ]:
const eventsData = eventsSnapshot.docs.map((doc) => ({
  uid: doc.id,
  name: doc.data().name ?? "",
  city: doc.data().city ?? "",
  address: doc.data().address ?? "",
  beginningDate: doc.data().beginningDate?.toDate()?.toISOString() ?? "",
  endDate: doc.data().endDate?.toDate()?.toISOString() ?? "",
  season: doc.data().season ?? "",
  link: doc.data().link ?? "",
  description: doc.data().description ?? "",
  deadline: doc.data().deadline?.toDate()?.toISOString() ?? "",
  organizers: doc.data().organizers ?? [],
  schedule: doc.data().schedule ?? {},
  price: doc.data().price ?? 0,
  motion: doc.data().motion ?? null, // <--- FIXED: Added motion mapping!
}));

// FIXED: Use a for...of loop so await actually pauses and executes sequentially
for (const event of eventsData) {
  console.log(`Processing event ${event.name} (${event.uid})`);

  const featuredProperties = [
    {
      icon: "i-ph-map-trifold-fill",
      label: "Kde",
      value: event.address,
    },
    {
      icon: "i-ph-watch-fill",
      label: "Kedy",
      value: formatRange(
        new Date(event.beginningDate),
        new Date(event.endDate),
      ),
    },
    {
      icon: "i-ph-coins-fill",
      label: "Koľko",
      value: event.price ? `${event.price} €` : "0 €",
    },
    {
      icon: "i-ph-warning-octagon-fill",
      label: "Deadline",
      value: event.deadline
        ? new Date(event.deadline).toLocaleDateString("sk-SK", {
          format: "short",
        })
        : "",
    },
    event.motion && {
      icon: "i-ph-quotes-fill",
      label: "Téza",
      value: event.motion,
    },
  ].filter(Boolean);

  // Remap the schedule
  const schedule = {
    days: event.schedule?.days?.map((day: any, index: number) => {
      const dayDate = new Date(event.beginningDate);
      dayDate.setDate(dayDate.getDate() + index);
      return {
        date: dayDate.toISOString(),
        schedule: day.schedule?.map((item: any, itemIndex: number) => {
          const previousDurations = day.schedule
            .slice(0, itemIndex)
            .reduce((acc: number, curr: any) => acc + curr.duration, 0);
          const itemBeginning = new Date(dayDate);
          itemBeginning.setMinutes(
            itemBeginning.getMinutes() + previousDurations,
          );
          return {
            ...item,
            text: item.name,
            name: undefined,
            beginning: itemBeginning.toISOString(),
          };
        }),
      };
    }),
  };

  let generatedEventId: number | null = null;

  try {
    const result = await client.query(
      'INSERT INTO events (slug, name, type, description, beginning, "end", target_league, target_region, place, featured_properties, schedule, registration_config) VALUES ($1, $2, $3, $4, $5, $6, $7, $8, $9, $10, $11, $12) RETURNING id',
      [
        event.uid,
        event.name + " " + event.season,
        event.motion ? "tournament" : "workshop",
        event.description,
        event.beginningDate,
        event.endDate,
        getTargetLeague(event.uid),
        getTargetRegion(event.name),
        event.city,
        JSON.stringify(featuredProperties),
        JSON.stringify(schedule),
        JSON.stringify({
          deadline: event.deadline ? event.deadline.split("T")[0] : null,
          href: event.link,
        }),
      ],
    );

    generatedEventId = result.rows[0].id;
    console.log(`Inserted event ${event.name} with ID ${generatedEventId}`);
  } catch (error) {
    console.error(`Error inserting event ${event.name}:`, error);
    continue; // Skip organizers if the event insert failed
  }

  // If the event has organizers, insert them sequentially as well
  if (event.organizers && Array.isArray(event.organizers)) {
    for (const organizer of event.organizers) {
      const user = usersXData.find((u) => u.uid === organizer);
      if (user && user.id) {
        try {
          await client.query(
            "INSERT INTO event_organizers (event_id, user_id) VALUES ($1, $2)",
            [generatedEventId, user.id],
          );
          console.log(
            `Inserted organizer ${user.name} ${user.surname} for event ${event.name}`,
          );
        } catch (error) {
          console.error(
            `Error inserting event organizer for ${event.name}:`,
            error,
          );
        }
      } else {
        console.error(
          `Error: Could not find user with uid ${organizer} for event ${event.name}`,
        );
      }
    }
  }
}


In [ ]:
// Close the pglite client
await client.close();
